### GRU Model

In [10]:
import numpy as np
import pandas as pd
import pickle
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, GRU, Dense,Dropout,BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

### load the datasets

In [11]:
X_train_padded = np.load("../models/X_train_padded.npy")
X_val_padded = np.load("../models/X_val_padded.npy")
X_test_padded = np.load("../models/X_test_padded.npy")
y_train = np.load("../models/y_train.npy")
y_val = np.load("../models/y_val.npy")
y_test = np.load("../models/y_test.npy")

sequence_length = X_train_padded.shape[1]
vocab_size = 20000

print(X_train_padded.shape)
print(X_val_padded.shape)
print(X_test_padded.shape)

(34705, 200)
(7439, 200)
(7438, 200)


In [12]:
gru_results = []

def evaluate_gru(model, experiment_name, X_test_data=None):

    # Use 200-token test data by default
    if X_test_data is None:
        X_test_data = X_test_padded

    # Prediction probabilities
    y_prob = model.predict(
        X_test_data,
        verbose=0
    ).ravel()

    # Convert probabilities to classes
    y_pred = (y_prob >= 0.5).astype(int)

    # Evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    # Store results
    gru_results.append({
        "Experiment": experiment_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    })

    # Display results
    print(f"\n{experiment_name}")
    print("-" * 40)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")

### Baseline GRU

In [10]:
gru_model = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    GRU(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,601,473 (9.92 MB)

 Trainable params: 2,601,473 (9.92 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
history_gru = gru_model.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 18s 13ms/step - accuracy: 0.5314 - loss: 0.6831 - val_accuracy: 0.5245 - val_loss: 0.6980
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - accuracy: 0.8519 - loss: 0.3291 - val_accuracy: 0.8942 - val_loss: 0.2573
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - accuracy: 0.9460 - loss: 0.1484 - val_accuracy: 0.8912 - val_loss: 0.2858
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - accuracy: 0.9789 - loss: 0.0677 - val_accuracy: 0.8808 - val_loss: 0.3703
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - accuracy: 0.9916 - loss: 0.0281 - val_accuracy: 0.8820 - val_loss: 0.5402


In [12]:
evaluate_gru(
    gru_model,
    "GRU - Baseline"
)
gru_model.save("/content/gru_baseline.keras")


GRU - Baseline
----------------------------------------
Accuracy : 0.8767
Precision: 0.8659
Recall   : 0.8926
F1 Score : 0.8790
ROC-AUC  : 0.9458


### Embedding Dimension

In [13]:
gru_embedding = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=256
    ),

    GRU(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

gru_embedding.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)



In [14]:
history_embedding = gru_embedding.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 21s 18ms/step - accuracy: 0.5139 - loss: 0.6918 - val_accuracy: 0.5256 - val_loss: 0.6824
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 24s 22ms/step - accuracy: 0.6858 - loss: 0.5180 - val_accuracy: 0.8814 - val_loss: 0.2946
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 32s 14ms/step - accuracy: 0.9267 - loss: 0.1915 - val_accuracy: 0.8886 - val_loss: 0.2804
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - accuracy: 0.9757 - loss: 0.0735 - val_accuracy: 0.8829 - val_loss: 0.4087
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - accuracy: 0.9927 - loss: 0.0265 - val_accuracy: 0.8767 - val_loss: 0.5393


In [15]:
evaluate_gru(
    gru_embedding,
    "GRU - Embedding 256"
)

gru_embedding.save("/content/gru_embedding_256.keras")


GRU - Embedding 256
----------------------------------------
Accuracy : 0.8759
Precision: 0.8507
Recall   : 0.9129
F1 Score : 0.8807
ROC-AUC  : 0.9408


### Sequential Length

In [13]:
with open("../models/tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)

vocab_size = len(tokenizer.word_index) + 1

In [14]:
X_train_text = pd.read_pickle("../models/X_train_text.pkl")
X_val_text = pd.read_pickle("../models/X_val_text.pkl")
X_test_text = pd.read_pickle("../models/X_test_text.pkl")

print(X_train_text.shape)
print(X_val_text.shape)
print(X_test_text.shape)

(34705,)
(7439,)
(7438,)


In [15]:
X_train_sequences = tokenizer.texts_to_sequences(X_train_text)
X_val_sequences = tokenizer.texts_to_sequences(X_val_text)
X_test_sequences = tokenizer.texts_to_sequences(X_test_text)

In [17]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_padded_300 = pad_sequences(
    X_train_sequences,
    maxlen=300,
    padding="post",
    truncating="post"
)

X_val_padded_300 = pad_sequences(
    X_val_sequences,
    maxlen=300,
    padding="post",
    truncating="post"
)

X_test_padded_300 = pad_sequences(
    X_test_sequences,
    maxlen=300,
    padding="post",
    truncating="post"
)

In [19]:
gru_sequence = Sequential([
    Input(shape=(300,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    GRU(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

gru_sequence.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


W0000 00:00:1787645513.962093  216651 cpu_allocator_impl.cc:82] Allocation of 43966976 exceeds 10% of free system memory.


In [20]:
history_sequence = gru_sequence.fit(
    X_train_padded_300,
    y_train,
    validation_data=(X_val_padded_300, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 256s 234ms/step - accuracy: 0.5002 - loss: 0.6927 - val_accuracy: 0.5110 - val_loss: 0.6897
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 259s 239ms/step - accuracy: 0.6182 - loss: 0.5904 - val_accuracy: 0.8726 - val_loss: 0.3132
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 256s 236ms/step - accuracy: 0.9104 - loss: 0.2263 - val_accuracy: 0.8886 - val_loss: 0.2693
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 259s 239ms/step - accuracy: 0.9666 - loss: 0.0990 - val_accuracy: 0.8853 - val_loss: 0.3291
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 260s 239ms/step - accuracy: 0.9900 - loss: 0.0349 - val_accuracy: 0.8808 - val_loss: 0.4172


In [22]:
evaluate_gru(
    gru_sequence,
    "GRU - Sequence Length 300",
    X_test_padded_300
)
gru_sequence.save("../models/gru_sequence_300.keras")


GRU - Sequence Length 300
----------------------------------------
Accuracy : 0.8756
Precision: 0.8660
Recall   : 0.8899
F1 Score : 0.8778
ROC-AUC  : 0.9462


### Hidden Units

In [36]:
gru_hidden = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    GRU(
        128,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

gru_hidden.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [37]:
history_hidden = gru_hidden.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 21s 18ms/step - accuracy: 0.5126 - loss: 0.6915 - val_accuracy: 0.5161 - val_loss: 0.6901
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 19ms/step - accuracy: 0.7905 - loss: 0.3991 - val_accuracy: 0.8918 - val_loss: 0.2666
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.9423 - loss: 0.1598 - val_accuracy: 0.8906 - val_loss: 0.2773
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 27s 24ms/step - accuracy: 0.9792 - loss: 0.0661 - val_accuracy: 0.8812 - val_loss: 0.3700
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 34s 18ms/step - accuracy: 0.9924 - loss: 0.0272 - val_accuracy: 0.8787 - val_loss: 0.4564


In [38]:
evaluate_gru(
    gru_hidden,
    "GRU - Hidden Units 128"
)

gru_hidden.save("/content/gru_hidden_128.keras")


GRU - Hidden Units 128
----------------------------------------
Accuracy : 0.8742
Precision: 0.8701
Recall   : 0.8808
F1 Score : 0.8754
ROC-AUC  : 0.9438


### Dropout

In [39]:
gru_dropout = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    GRU(
        64,
        activation="tanh"
    ),

    Dropout(0.3),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.3),

    Dense(
        1,
        activation="sigmoid"
    )
])

gru_dropout.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [40]:
history_dropout = gru_dropout.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 22s 18ms/step - accuracy: 0.5028 - loss: 0.6931 - val_accuracy: 0.5100 - val_loss: 0.6927
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 19s 18ms/step - accuracy: 0.6144 - loss: 0.6031 - val_accuracy: 0.8693 - val_loss: 0.3079
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 19s 18ms/step - accuracy: 0.9075 - loss: 0.2364 - val_accuracy: 0.8946 - val_loss: 0.2545
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - accuracy: 0.9612 - loss: 0.1122 - val_accuracy: 0.8781 - val_loss: 0.3430
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 19s 18ms/step - accuracy: 0.9848 - loss: 0.0474 - val_accuracy: 0.8766 - val_loss: 0.3957


In [41]:
evaluate_gru(
    gru_dropout,
    "GRU - Dropout"
)

gru_dropout.save("/content/gru_dropout.keras")


GRU - Dropout
----------------------------------------
Accuracy : 0.8720
Precision: 0.8441
Recall   : 0.9137
F1 Score : 0.8775
ROC-AUC  : 0.9451


### Different Optimizer

In [42]:
gru_rmsprop = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    GRU(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

gru_rmsprop.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [43]:

history_rmsprop = gru_rmsprop.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 17s 15ms/step - accuracy: 0.5012 - loss: 0.6934 - val_accuracy: 0.5128 - val_loss: 0.6923
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 16s 15ms/step - accuracy: 0.5247 - loss: 0.6860 - val_accuracy: 0.5374 - val_loss: 0.7901
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - accuracy: 0.6005 - loss: 0.6151 - val_accuracy: 0.8269 - val_loss: 0.4079
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - accuracy: 0.8860 - loss: 0.2868 - val_accuracy: 0.8887 - val_loss: 0.2790
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - accuracy: 0.9238 - loss: 0.2019 - val_accuracy: 0.8937 - val_loss: 0.2822


In [44]:

evaluate_gru(
    gru_rmsprop,
    "GRU - RMSprop"
)

gru_rmsprop.save("/content/gru_rmsprop.keras")


GRU - RMSprop
----------------------------------------
Accuracy : 0.8928
Precision: 0.9064
Recall   : 0.8770
F1 Score : 0.8915
ROC-AUC  : 0.9577


### Batch Normalization

In [45]:
gru_batchnorm = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    GRU(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    BatchNormalization(),

    Dense(
        1,
        activation="sigmoid"
    )
])

gru_batchnorm.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [47]:

history_batchnorm = gru_batchnorm.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6316 - loss: 0.5733 - val_accuracy: 0.6065 - val_loss: 1.1555
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 22s 20ms/step - accuracy: 0.9055 - loss: 0.2382 - val_accuracy: 0.8953 - val_loss: 0.2567
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 19s 18ms/step - accuracy: 0.9543 - loss: 0.1275 - val_accuracy: 0.8528 - val_loss: 0.3970
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.9788 - loss: 0.0625 - val_accuracy: 0.8771 - val_loss: 0.4020
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 19s 18ms/step - accuracy: 0.9894 - loss: 0.0330 - val_accuracy: 0.8599 - val_loss: 0.5995


In [48]:

evaluate_gru(
    gru_batchnorm,
    "GRU - Batch Normalization"
)

gru_batchnorm.save("/content/gru_batchnorm.keras")


GRU - Batch Normalization
----------------------------------------
Accuracy : 0.8588
Precision: 0.9173
Recall   : 0.7900
F1 Score : 0.8489
ROC-AUC  : 0.9453


### Learning Rate

In [49]:
gru_learning_rate = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    GRU(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

gru_learning_rate.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [51]:

history_learning_rate = gru_learning_rate.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)



Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 23s 19ms/step - accuracy: 0.5017 - loss: 0.6931 - val_accuracy: 0.5118 - val_loss: 0.6925
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 21s 19ms/step - accuracy: 0.5180 - loss: 0.6882 - val_accuracy: 0.5214 - val_loss: 0.6880
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 18s 17ms/step - accuracy: 0.5513 - loss: 0.6603 - val_accuracy: 0.5271 - val_loss: 0.6766
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 19s 18ms/step - accuracy: 0.5678 - loss: 0.6285 - val_accuracy: 0.5338 - val_loss: 0.6840
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 18s 17ms/step - accuracy: 0.5750 - loss: 0.6153 - val_accuracy: 0.5276 - val_loss: 0.7060


In [52]:
evaluate_gru(
    gru_learning_rate,
    "GRU - Learning Rate 0.0001"
)

gru_learning_rate.save("/content/gru_learning_rate.keras")


GRU - Learning Rate 0.0001
----------------------------------------
Accuracy : 0.5289
Precision: 0.5164
Recall   : 0.9668
F1 Score : 0.6732
ROC-AUC  : 0.5875


### Batch Size

In [53]:
gru_batch_size = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    GRU(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

gru_batch_size.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [54]:

history_batch_size = gru_batch_size.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=64,
    verbose=1
)


Epoch 1/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 18ms/step - accuracy: 0.7231 - loss: 0.4835 - val_accuracy: 0.8755 - val_loss: 0.3072
Epoch 2/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9191 - loss: 0.2094 - val_accuracy: 0.8844 - val_loss: 0.2958
Epoch 3/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9608 - loss: 0.1166 - val_accuracy: 0.8818 - val_loss: 0.3412
Epoch 4/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9811 - loss: 0.0631 - val_accuracy: 0.8738 - val_loss: 0.4169
Epoch 5/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9888 - loss: 0.0361 - val_accuracy: 0.8742 - val_loss: 0.4660


In [55]:

evaluate_gru(
    gru_learning_rate,
    "GRU - Batch Size 64"
)

gru_learning_rate.save("/content/GRU_batch_size_64.keras")


GRU - Batch Size 64
----------------------------------------
Accuracy : 0.8677
Precision: 0.8514
Recall   : 0.8920
F1 Score : 0.8713
ROC-AUC  : 0.9351


### Early Stopping

In [56]:
gru_early_stopping = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    GRU(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

gru_early_stopping.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)


In [57]:

history_early_stopping = gru_early_stopping.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)



Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.5069 - loss: 0.6932 - val_accuracy: 0.5192 - val_loss: 0.6899
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 18s 17ms/step - accuracy: 0.7171 - loss: 0.4926 - val_accuracy: 0.8859 - val_loss: 0.2878
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 19s 18ms/step - accuracy: 0.9292 - loss: 0.1880 - val_accuracy: 0.8863 - val_loss: 0.2802
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - accuracy: 0.9741 - loss: 0.0817 - val_accuracy: 0.8826 - val_loss: 0.3432
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 19s 18ms/step - accuracy: 0.9897 - loss: 0.0355 - val_accuracy: 0.8787 - val_loss: 0.4636


In [58]:
evaluate_gru(
    gru_early_stopping,
    "GRU - Early Stopping"
)

gru_early_stopping.save(
    "/content/gru_early_stopping.keras"
)


GRU - Early Stopping
----------------------------------------
Accuracy : 0.8855
Precision: 0.8888
Recall   : 0.8821
F1 Score : 0.8855
ROC-AUC  : 0.9517


### Learning Rate Scheduling

In [59]:
gru_scheduler = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    GRU(
        64,
        activation="tanh"
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

gru_scheduler.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6
)


In [60]:

history_scheduler = gru_scheduler.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    callbacks=[reduce_lr],
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.5167 - loss: 0.6887 - val_accuracy: 0.7688 - val_loss: 0.5562 - learning_rate: 0.0010
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 23s 21ms/step - accuracy: 0.8757 - loss: 0.3040 - val_accuracy: 0.8892 - val_loss: 0.2620 - learning_rate: 0.0010
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 36s 17ms/step - accuracy: 0.9508 - loss: 0.1370 - val_accuracy: 0.8860 - val_loss: 0.2932 - learning_rate: 0.0010
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - accuracy: 0.9873 - loss: 0.0458 - val_accuracy: 0.8813 - val_loss: 0.3840 - learning_rate: 5.0000e-04
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 18s 17ms/step - accuracy: 0.9963 - loss: 0.0172 - val_accuracy: 0.8808 - val_loss: 0.5057 - learning_rate: 2.5000e-04


In [61]:

evaluate_gru(
    gru_scheduler,
    "GRU - Learning Rate Scheduling"
)

gru_scheduler.save(
    "/content/GRU_learning_rate_scheduler.keras"
)


GRU - Learning Rate Scheduling
----------------------------------------
Accuracy : 0.8778
Precision: 0.8957
Recall   : 0.8561
F1 Score : 0.8755
ROC-AUC  : 0.9478


### Recurrent Dropout

In [62]:
gru_recurrent_dropout = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    GRU(
        64,
        activation="tanh",
        recurrent_dropout=0.3
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

gru_recurrent_dropout.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)



In [64]:
history_recurrent_dropout = gru_recurrent_dropout.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 610s 562ms/step - accuracy: 0.5198 - loss: 0.6890 - val_accuracy: 0.6931 - val_loss: 0.6702
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 614s 565ms/step - accuracy: 0.8542 - loss: 0.3476 - val_accuracy: 0.8898 - val_loss: 0.2725
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 615s 567ms/step - accuracy: 0.9407 - loss: 0.1686 - val_accuracy: 0.8911 - val_loss: 0.2897
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 605s 557ms/step - accuracy: 0.9735 - loss: 0.0850 - val_accuracy: 0.8817 - val_loss: 0.3450
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 608s 561ms/step - accuracy: 0.9873 - loss: 0.0454 - val_accuracy: 0.8771 - val_loss: 0.4638


In [65]:

evaluate_gru(
    gru_recurrent_dropout,
    "GRU - Recurrent Dropout"
)

gru_recurrent_dropout.save(
    "/content/gru_recurrent_dropout.keras"
)


GRU - Recurrent Dropout
----------------------------------------
Accuracy : 0.8816
Precision: 0.8879
Recall   : 0.8744
F1 Score : 0.8811
ROC-AUC  : 0.9435


In [66]:
gru_results_df = pd.DataFrame(gru_results)

gru_results_df

,Experiment,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,GRU - Baseline,0.876714,0.865904,0.892580,0.879040,0.945839
1,GRU - Embedding 256,0.875908,0.850724,0.912939,0.880734,0.940829
2,GRU - Hidden Units 128,0.874160,0.870071,0.880793,0.875399,0.943837
3,GRU - Dropout,0.872009,0.844098,0.913742,0.877541,0.945119
4,GRU - RMSprop,0.892848,0.906423,0.877043,0.891491,0.957674
5,GRU - Batch Normalization,0.858833,0.917263,0.789981,0.848877,0.945253
6,GRU - Learning Rate 0.0001,0.528906,0.516383,0.966783,0.673195,0.587538
7,GRU - Batch Size 64,0.867706,0.851445,0.892044,0.871272,0.935084
8,GRU - Early Stopping,0.885453,0.888799,0.882132,0.885453,0.951678
9,GRU - Learning Rate Scheduling,0.877790,0.895740,0.856148,0.875497,0.947842


In [67]:
gru_results_df.to_csv("/content/gru_comparison_table.csv",index=False)